# Page 11 — "The Keystone", measured: training vs scale vs test-time compute

One family, one grader, one question: at matched inference FLOPs, what actually buys
accuracy — **parameters**, **samples**, or **training**? This notebook does not open with new
generation. Its arm 0 re-derives the accuracy-against-FLOPs family chart from the runs pages
6, 7 and 8 already published, under one rule: **every recomputed number must equal the one
those pages shipped** (gate 1). Only then do the new arms spend anything:

- **Gate 2** (`QUICK`) — a 20-item smoke test of `Qwen2.5-Math-1.5B-Instruct` that decides
  whether arm B runs at all, and measures its chain length for the FLOPs-parity accounting.
- **Arm A** (`FULL`) — the easy band: MATH-500 levels 1–3, where the published crossing must
  appear if it appears anywhere on this instrument.
- **Arm B** (`FULL`) — the specialist: the Math model, trained *by* the reasoning loop
  (rejection-sampled rationales, then GRPO against a correctness verifier), against the
  general 7B's greedy answer.

Outcome branches are pre-registered in the spec **before** the run; every branch ships.

Writes `out/page_11/`: `traces.jsonl` (new arms only), `derived.json`, `manifest.json`.

All rationale lives in [`docs/page_11_refactor_spec.md`](../docs/page_11_refactor_spec.md).

## How to run this

| Mode | What runs | Needs | Time |
|---|---|---|---|
| `REPLAY` | arm 0: the derivation and gate 1, nothing generated | pages 6/7/8's `out/`, no GPU, no weights | ~10 min |
| `QUICK` | arm 0 + gate 2 (Math-1.5B greedy, 20 items) | ~4 GB GPU, or a patient CPU | +~20 min |
| `FULL` | arm 0 + gate 2 + arms A and B | a 16 GB GPU | hours |

**Inputs.** Arm 0 reads `out/page_06/`, `out/page_07/` and `out/page_08/` — run those
notebooks, or unpack their published zips so their `out/` directories land beside this file.
Each missing run degrades to a printed skip, but gate 1 passes only when all three are
present, and nothing downstream of gate 1 may be trusted before it passes.

**The ports below must match byte for byte.** Gate 1 compares this notebook's recomputed
aggregates against each upstream run's own `derived.json`; any drift means the deck would
show one measurement with two values.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU) | "QUICK" (+gate 2) | "FULL" (+arms A and B)
SEED = 123                # pages 6-8's seed: the item draws must be their draws

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"          # Act 2's shared instrument
FAMILY = ["Qwen/Qwen2.5-1.5B-Instruct",       # the generalist ladder, pages 6-8's roster
          "Qwen/Qwen2.5-3B-Instruct",
          "Qwen/Qwen2.5-7B-Instruct"]
BIG = "Qwen/Qwen2.5-7B-Instruct"              # the target: its one greedy answer

# Arm B. Same lineage, math-specialised branch: continued pretraining on ~1T math tokens,
# SFT on rejection-sampled rationales, then GRPO with a rule-based correctness verifier in
# the reward (Qwen2.5-Math report). Provenance and the bundle-attribution caveat: spec §2.
MATH_MODEL = "Qwen/Qwen2.5-Math-1.5B-Instruct"
# Its model card's CoT system prompt. The user turn stays ch03's render_prompt, identical
# to the family's, so the comparison varies the model and not the question.
MATH_SYSTEM = "Please reason step by step, and put your final answer within \\boxed{}."

# The two bands. Hard = pages 6-8's exact 100 items (asserted below against page 7's
# manifest). Easy = 50 fresh L1-3 items, same draw rule: the spec's mandatory arm A
# (spec §4.2 defends why L4/5 alone cannot decide the crossing).
HARD_LEVELS, HARD_N = (4, 5), 100
EASY_LEVELS, EASY_N = (1, 2, 3), 50

# Decoding, page 7's exactly: the new arms must be draws from the same process.
POOL_N = 16               # chains per item on the vote arms
T_VOTE = 0.3              # the pile temperature page 7's diversity family chose
TOP_P = 0.95
CHAIN_CAP = 2048          # a safety cap, not a budget; truncation is reported, not hidden
GROUP = 8                 # chains per generate call
ANSWER_ALLOWANCE = 32     # page 7's forced-commit allowance for a capped chain
N_AXIS = [1, 2, 4, 8, 16]
N_RESAMPLES = 64          # per (item, N): page 7's estimator, asserted against its manifest

# Gate 2, pre-registered in the spec before any Math-model output was seen: the floor the
# specialist must clear on 20 hard-band items (7B is at 60%) for arm B to run, else branch 5.
GATE2_ITEMS = 20
GATE2_MIN = 0.45

# FLOPs accounting, one rule for every point on the chart: 2 * params * tokens_generated,
# measured tokens rather than caps (the forward-pass approximation; parameter ratios are
# NOT quoted as FLOPs ratios anywhere). A best-of point adds its judge's pass at the same
# rule. Prompt-prefix FLOPs are omitted on both sides and said so in the caption.
FLOPS = lambda params, tokens: 2.0 * params * tokens

# REPLAY inputs when the sibling runs are not on disk: the published traces of pages 6-8.
# None means "already in out/", which is how the zips unpack.
PAGE_06_TRACES_URL = None
PAGE_07_TRACES_URL = None
PAGE_08_TRACES_URL = None
PAGE_08_SCORES_URL = None

print(f"mode={RUN_MODE}  primary={PRIMARY}")
print(f"target={BIG}  specialist={MATH_MODEL}")
print(f"bands: hard L{HARD_LEVELS} n={HARD_N} (pages 6-8's), easy L{EASY_LEVELS} n={EASY_N}")

In [ ]:
import json
import math
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

# Shared plumbing, shipped beside this notebook.
import harness

# The inherited grading harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_11")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

UP = {n: Path(f"out/page_{n:02d}") for n in (6, 7, 8)}


def read_traces(path):
    """Page 7's reader: every record, last write per key winning, so a widened group is
    one chain rather than two. Page 6 keys by budget and has no sample axis, so the key
    is whatever axes the file carries."""
    latest = {}
    for line in Path(path).open(encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        latest[(r["run_id"], r["model_id"], r.get("config_id", r.get("budget")),
                r["problem_id"], r.get("sample_index", 0))] = r
    return list(latest.values())


def fetch_if_missing(path, url, label):
    if url and not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"fetching {label} from {url}")
        path.write_text(requests.get(url, timeout=600).text, encoding="utf-8")


mean, median = harness.mean, harness.median

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## 1. The two item bands

The hard band must be **pages 6–8's exact draw** — same seed, same rule — or every
cross-check below compares different populations. The easy band is a fresh draw by the same
rule from levels 1–3, disjoint from the hard band by construction.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")


def draw_band(levels, n):
    band = sorted((r for r in math500 if int(r["level"]) in levels),
                  key=lambda r: r["unique_id"])
    random.Random(SEED).shuffle(band)
    assert len(band) >= n, f"levels {levels} hold only {len(band)} items, asked for {n}"
    return band[:n]


items_hard = draw_band(HARD_LEVELS, HARD_N)
items_easy = draw_band(EASY_LEVELS, EASY_N)

HARD_HASH = harness.sha256_of(r["unique_id"] for r in items_hard)
EASY_HASH = harness.sha256_of(r["unique_id"] for r in items_easy)
assert not ({r["unique_id"] for r in items_hard} & {r["unique_id"] for r in items_easy})

print(f"hard band  L{HARD_LEVELS}  n={len(items_hard)}  sha256 {HARD_HASH[:12]}")
print(f"easy band  L{EASY_LEVELS}  n={len(items_easy)}  sha256 {EASY_HASH[:12]}")

p7_manifest_path = UP[7] / "manifest.json"
if p7_manifest_path.exists():
    _p7m = json.loads(p7_manifest_path.read_text())
    assert _p7m["items"]["item_set_sha256"] == HARD_HASH, (
        "this notebook's hard band is NOT page 7's item draw; every gate below would "
        "compare different populations. Match SEED, HARD_LEVELS and HARD_N.")
    print("-> hard band is page 7's draw, verified against its manifest")
else:
    print("-> out/page_07/manifest.json not found; the draw is asserted only by rule")

## 2. The prior runs on disk (arm 0's inputs)

Three runs, one instrument. Parameter counts come from page 6's `derived.json` — the counts
the loaded checkpoints reported — not from names.

In [ ]:
SRC = {}          # n -> dict(traces=..., derived=..., manifest=...)
for n in (6, 7, 8):
    d = UP[n]
    fetch_if_missing(d / "traces.jsonl",
                     {6: PAGE_06_TRACES_URL, 7: PAGE_07_TRACES_URL,
                      8: PAGE_08_TRACES_URL}[n], f"page {n} traces")
    if not (d / "traces.jsonl").exists() or not (d / "derived.json").exists():
        print(f"page {n}: out/page_{n:02d}/ incomplete -- its gates and points are SKIPPED")
        continue
    SRC[n] = dict(traces=read_traces(d / "traces.jsonl"),
                  derived=json.loads((d / "derived.json").read_text()),
                  manifest=json.loads((d / "manifest.json").read_text()))
    rid = SRC[n]["manifest"].get("run_id") or SRC[n]["manifest"].get("score_id")
    print(f"page {n}: {len(SRC[n]['traces']):,} records, run {rid}")

fetch_if_missing(UP[8] / "scores.jsonl", PAGE_08_SCORES_URL, "page 8 scores")

# The estimator this notebook re-runs must be the one that made the numbers it checks
# against, so the knobs are asserted rather than trusted.
if 7 in SRC:
    _pile = SRC[7]["manifest"]["pile"]
    assert _pile["n_resamples"] == N_RESAMPLES and _pile["n_axis"] == N_AXIS
    assert SRC[7]["manifest"]["items"]["seed"] == SEED
if 8 in SRC:
    _sub = SRC[8]["manifest"]["subsampling"]
    assert _sub["seed"] == SEED and _sub["n_resamples"] == N_RESAMPLES

# Exact parameter counts, from the checkpoints page 6 loaded.
PARAMS = {}
if 6 in SRC:
    for c in SRC[6]["derived"]["figure_2"]["curves"]:
        PARAMS[c["model_id"]] = c["params"]
    for mid, p in PARAMS.items():
        print(f"  {mid.split('/')[-1]:<24} {p:,} params")

## 3. The vote and the selectors, ported

Ported from pages 7 and 8 **verbatim** (same names, same order of operations): `vote_key` /
`cluster` / `tally` / `aggregate` and `subsets_for` are page 7's; `plurality` /
`sel_majority` / `by_score` are page 8's. Gate 1 exists to catch this port drifting.

In [ ]:
NO_ANSWER = None


def vote_key(text):
    """What a chain contributes to the tally, or None if it contributed nothing."""
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ_CACHE = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ_CACHE:
        try:
            # grade_answer is not documented as symmetric, so ask it both ways.
            _EQ_CACHE[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ_CACHE[k] = False
    return _EQ_CACHE[k]


def cluster(keys):
    """Answer strings grouped into equivalence classes, in first-seen order."""
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


def tally(records):
    """One problem, N chains -> the row the figures read. Page 7's, verbatim."""
    gt = records[0]["ground_truth"]
    labels, _ = cluster([vote_key(r["trace"]) for r in records])

    counts = Counter([l for l in labels if l is not NO_ANSWER])
    n = len(records)
    ranked = counts.most_common()
    top = ranked[0][1] if ranked else 0
    winners = [a for a, c in ranked if c == top]
    tie = len(winners) > 1
    plurality_answer = winners[0] if (winners and not tie) else None
    runner_up = next((c for _, c in ranked[1:]), 0)
    total = sum(counts.values())
    entropy = -sum((c / total) * math.log2(c / total) for c in counts.values()) if total else 0.0

    return dict(
        n_chains=n,
        distinct_answers=len(counts),
        answer_entropy=entropy,
        entropy_normalised=(entropy / math.log2(n)) if n > 1 and total else 0.0,
        plurality_answer=plurality_answer,
        plurality_votes=top,
        plurality_margin=top - runner_up,
        tie=tie,
        plurality_correct=bool(plurality_answer is not None
                               and grade_answer(plurality_answer, gt)),
        any_correct=any(r["correct"] for r in records),
        first_chain_correct=bool(records[0]["correct"]),
        mean_chain_correct=mean([r["correct"] for r in records]),
        no_answer_rate=mean([l is NO_ANSWER for l in labels]),
        truncated_rate=mean([r["truncated"] for r in records]),
        tokens_generated=sum(r["tokens_generated"] for r in records),
        ranked=[(a, c) for a, c in ranked],
    )


def aggregate(rows):
    """Many problems, one configuration -> the point the curves plot. Page 7's, verbatim."""
    n = len(rows)
    k = sum(r["plurality_correct"] for r in rows)
    lo, hi = harness.wilson(k, n)
    return dict(
        n_items=n,
        plurality_accuracy=k / n, ci_lo=lo, ci_hi=hi,
        coverage=mean([r["any_correct"] for r in rows]),
        single_chain_accuracy=mean([r["mean_chain_correct"] for r in rows]),
        thrown_away=mean([r["any_correct"] and not r["plurality_correct"] for r in rows]),
        mean_distinct=mean([r["distinct_answers"] for r in rows]),
        mean_entropy=mean([r["answer_entropy"] for r in rows]),
        tie_rate=mean([r["tie"] for r in rows]),
        no_answer_rate=mean([r["no_answer_rate"] for r in rows]),
        truncated_rate=mean([r["truncated_rate"] for r in rows]),
        mean_tokens_generated=mean([r["tokens_generated"] for r in rows]),
    )


def subsets_for(pid, n, repeats, size):
    """Page 7's draws: which chains a point at N averages over. Exhaustive when the
    configuration holds exactly n."""
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def curve_point(samples_by_pid, items, n):
    """One point on page 7's N axis, its estimator verbatim."""
    per_item = []
    for p in items:
        rs = samples_by_pid.get(p["unique_id"])
        if not rs or len(rs) < n:
            continue
        ts = [tally([rs[i] for i in idx])
              for idx in subsets_for(p["unique_id"], n, N_RESAMPLES, len(rs))]
        per_item.append({k: mean([t[k] for t in ts]) for k in
                         ("plurality_correct", "any_correct", "tie", "mean_chain_correct",
                          "distinct_answers", "tokens_generated", "truncated_rate")})
    if not per_item:
        return None
    m = lambda k: mean([x[k] for x in per_item])
    return dict(n=n, n_items=len(per_item),
                accuracy=m("plurality_correct"), coverage=m("any_correct"),
                tie_rate=m("tie"), single_chain_accuracy=m("mean_chain_correct"),
                mean_distinct=m("distinct_answers"),
                mean_tokens=m("tokens_generated"),
                truncated_rate=m("truncated_rate"),
                n_draws=len(per_item) * N_RESAMPLES)


# --- page 8's selectors, verbatim -------------------------------------------------
def plurality(cands):
    """The class the most candidates agree on, or None on a tie or an empty tally."""
    labels, _ = cluster([c.get("vote_key") or vote_key(c["trace"]) for c in cands])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    mc = counts.most_common()
    if not mc:
        return None
    top = mc[0][1]
    winners = [a for a, c in mc if c == top]
    return winners[0] if len(winners) == 1 else None


def sel_majority(cands, rng):
    win = plurality(cands)
    if win is None:
        return None
    for i, c in enumerate(cands):
        k = c.get("vote_key") or vote_key(c["trace"])
        if k is not NO_ANSWER and equivalent(k, win):
            return i
    return None


def by_score(scores, field):
    """Highest score in `field` wins, ties to the lowest index. Page 8's rule."""
    def pick(cands, rng):
        def key(i):
            s = scores[(cands[i]["config_id"], cands[i]["problem_id"],
                        cands[i]["sample_index"])].get(field)
            return (-math.inf if s is None else s, -i)
        return max(range(len(cands)), key=key)
    return pick


print("vote and selectors ported; gate 1 below is the drift trap")

## 4. Gate 1a — page 6's greedy sweep, recomputed

Every `(checkpoint, budget)` cell of page 6's sweep: accuracy and measured spend, from its
raw traces, compared against its shipped `derived.json`. These are the family chart's greedy
points, and the 7B row at the top budget is **the target**.

In [ ]:
GATE1 = {}
P6_POINTS = {}

if 6 in SRC:
    by_cell = {}
    for r in SRC[6]["traces"]:
        if r["model_id"] in FAMILY and r["budget"] is not None:
            by_cell.setdefault((r["model_id"], r["budget"]), []).append(r)

    worst = 0.0
    rows = []
    for c in SRC[6]["derived"]["figure_2"]["curves"]:
        mid = c["model_id"]
        for pt in c["points"]:
            rs = by_cell.get((mid, pt["budget"]), [])
            acc = mean([r["correct"] for r in rs])
            tok = mean([r["tokens_generated"] for r in rs])
            worst = max(worst, abs(acc - pt["accuracy"]), abs(tok - pt["mean_tokens"]))
            P6_POINTS.setdefault(mid, {})[pt["budget"]] = dict(
                accuracy=acc, mean_tokens=tok, n=len(rs))
            rows.append((mid.split("/")[-1], pt["budget"], len(rs),
                         f"{acc:.0%}", f"{tok:,.0f}"))

    harness.table("page 6 recomputed, every cell",
                  ["checkpoint", "cap", "n", "solved", "mean spend"], rows)
    GATE1["page6"] = dict(reproduced=bool(worst < 1e-9), max_abs_drift=worst,
                          cells=len(rows))
    print(f"\nmax |recomputed - shipped| over {len(rows)} cells: {worst:.2e}"
          f"   -> {'REPRODUCED' if GATE1['page6']['reproduced'] else 'DRIFTED'}")
else:
    GATE1["page6"] = dict(reproduced=False, skipped=True)
    print("page 6 not on disk; gate 1a cannot run")

## 5. Gate 1b — page 7's pile, recomputed

The hand-off row (the vote, coverage and what the vote threw away, on the full pile) and
both pile curves (T=0.3 and the degenerate T=0 line), through the ported estimator with page
7's own seeded draws.

In [ ]:
P7_VOTE_CURVE = None

if 7 in SRC:
    want = {r["unique_id"] for r in items_hard}
    p7_raw = SRC[7]["traces"]
    p7_run = Counter(r["run_id"] for r in p7_raw).most_common(1)[0][0]
    p7_raw = [r for r in p7_raw if r["run_id"] == p7_run and r["model_id"] == PRIMARY
              and r["problem_id"] in want]

    S7 = {}
    for r in p7_raw:
        S7.setdefault(r["config_id"], {}).setdefault(r["problem_id"], []).append(r)
    for cid in S7:
        for pid in S7[cid]:
            S7[cid][pid].sort(key=lambda r: r["sample_index"])

    d7 = SRC[7]["derived"]
    POOL_CONFIG = d7["pool"]["config_id"]

    # -- the hand-off row: the full pile, first POOL_N chains, no subsampling ------
    rows7 = [tally(S7[POOL_CONFIG][p["unique_id"]][:POOL_N])
             for p in items_hard if p["unique_id"] in S7[POOL_CONFIG]]
    A = aggregate(rows7)
    ship = d7["handoff"]
    drift = {k: abs(A[k] - ship[k]) for k in
             ("plurality_accuracy", "coverage", "single_chain_accuracy",
              "thrown_away", "tie_rate")}
    print(f"the pile, {POOL_N} chains at T={ship['temperature']}, "
          f"{A['n_items']} problems   (shipped -> recomputed)")
    for k, d in drift.items():
        print(f"  {k:<24} {ship[k]:>8.4f} -> {A[k]:>8.4f}   |d|={d:.2e}")

    # -- both pile curves, every N ---------------------------------------------------
    worst = max(drift.values())
    n_pts = 0
    for entry in d7["figure_2"]["pile"]:
        cid = f"t{entry['temperature']:.2f}-c{ship['chain_cap']}"
        for pt in entry["points"]:
            ours = curve_point(S7[cid], items_hard, pt["n"])
            for k in ("accuracy", "coverage", "tie_rate", "single_chain_accuracy",
                      "mean_distinct", "mean_tokens", "truncated_rate"):
                worst = max(worst, abs(ours[k] - pt[k]))
            n_pts += 1
        if entry["temperature"] == ship["temperature"]:
            P7_VOTE_CURVE = [curve_point(S7[cid], items_hard, n) for n in N_AXIS]

    GATE1["page7"] = dict(reproduced=bool(worst < 1e-9), max_abs_drift=worst,
                          handoff=dict(A), curve_points=n_pts)
    print(f"\nmax |recomputed - shipped| over the hand-off and {n_pts} curve points: "
          f"{worst:.2e}   -> {'REPRODUCED' if GATE1['page7']['reproduced'] else 'DRIFTED'}")
else:
    GATE1["page7"] = dict(reproduced=False, skipped=True)
    print("page 7 not on disk; gate 1b cannot run")

## 6. Gate 1c — page 8's paired rung, recomputed

The rung the spec quotes: on page 8's own pool, one deterministic outcome per item —
candidate 0 as the baseline, the majority vote, and best-of-N under the trained verifier —
with the McNemar counts. Page 8's pool build (dedup, trim to common depth) is ported with it.

In [ ]:
P8_RUNG = None

if 8 in SRC and (UP[8] / "scores.jsonl").exists():
    d8 = SRC[8]["derived"]
    ship = d8["paired"]
    PAIRED_CONFIG = ship["config_id"]
    SCORE_ID = SRC[8]["manifest"]["score_id"]

    # page 8's load: majority run_id, dedup by key with the identical-trace assertion
    rows8 = SRC[8]["traces"]
    run8 = Counter(r["run_id"] for r in rows8).most_common(1)[0][0]
    rows8 = [r for r in rows8 if r["run_id"] == run8]
    mid8 = Counter(r["model_id"] for r in rows8).most_common(1)[0][0]
    rows8 = [r for r in rows8 if r["model_id"] == mid8]

    # page 8's pool build: sort by sample_index, drop shallow items, trim to common depth
    MIN_POOL = 8
    pools = {}
    for r in rows8:
        pools.setdefault(r["config_id"], {}).setdefault(r["problem_id"], []).append(r)
    for cid in pools:
        for pid in pools[cid]:
            pools[cid][pid].sort(key=lambda r: r["sample_index"])
    deep = {p: v for p, v in pools[PAIRED_CONFIG].items() if len(v) >= MIN_POOL}
    depth = min(len(v) for v in deep.values())
    pool = {p: v[:depth] for p, v in deep.items()}
    ids8 = sorted(pool)
    assert depth == ship["n"] and len(ids8) == ship["n_items"], (
        f"pool shape ({depth}, {len(ids8)}) is not the shipped ({ship['n']}, "
        f"{ship['n_items']}); a different run is on disk")

    scores8 = {}
    for line in (UP[8] / "scores.jsonl").open(encoding="utf-8"):
        if not line.strip():
            continue
        s = json.loads(line)
        if s.get("score_id") != SCORE_ID or s["config_id"] != PAIRED_CONFIG:
            continue
        if "scorer" in s and s.get("score") is not None:
            scores8.setdefault((s["config_id"], s["problem_id"], s["sample_index"]),
                               {})[f"v:{s['scorer']}"] = s["score"]

    # the trained verifier = the best-of row the shipped paired block carries
    VERIFIER = next(r["selector"] for r in ship["vs_one_chain"]
                    if "prm" in r["selector"])
    v_field = "v:" + VERIFIER[VERIFIER.find("(") + 1:-1]
    n_scored = sum(1 for v in scores8.values() if v_field in v)
    assert n_scored == len(ids8) * depth, (
        f"{n_scored} {v_field} scores for a {len(ids8)}x{depth} pool; scores.jsonl is "
        f"missing rows for this run")

    _rng = random.Random(harness.sha256_of([str(SEED), "paired"])[:15])
    sel_best = by_score(scores8, v_field)
    outcome = {
        "one chain": {pid: bool(pool[pid][0]["correct"]) for pid in ids8},
        "majority": {pid: (lambda i: bool(i is not None and pool[pid][i]["correct"]))
                     (sel_majority(pool[pid], _rng)) for pid in ids8},
        VERIFIER: {pid: (lambda i: bool(i is not None and pool[pid][i]["correct"]))
                   (sel_best(pool[pid], _rng)) for pid in ids8},
    }

    base = outcome["one chain"]
    worst = abs(mean(list(base.values())) - ship["baseline_accuracy"])
    print(f"best-of-{depth} against one chain, {len(ids8)} problems at {PAIRED_CONFIG}\n")
    print(f"  {'rule':<26}{'shipped':>9}{'ours':>9}{'fixed':>7}{'broke':>7}{'p':>10}")
    print(f"  {'one chain (baseline)':<26}{ship['baseline_accuracy']:>9.1%}"
          f"{mean(list(base.values())):>9.1%}")
    for name in ("majority", VERIFIER):
        srow = next(r for r in ship["vs_one_chain"] if r["selector"] == name)
        broke, fixed, pv = harness.mcnemar(base, outcome[name])
        acc = mean(list(outcome[name].values()))
        worst = max(worst, abs(acc - srow["accuracy"]), abs(pv - srow["p"]),
                    abs(fixed - srow["fixed"]), abs(broke - srow["broke"]))
        print(f"  {name:<26}{srow['accuracy']:>9.1%}{acc:>9.1%}{fixed:>7}{broke:>7}"
              f"{pv:>10.4f}")
        if name == VERIFIER:
            P8_RUNG = dict(config_id=PAIRED_CONFIG, n=depth, selector=name,
                           accuracy=acc, fixed=fixed, broke=broke, p=pv,
                           baseline_accuracy=mean(list(base.values())),
                           majority_accuracy=mean(list(outcome["majority"].values())),
                           mean_tokens=mean([sum(r["tokens_generated"] for r in pool[pid])
                                             for pid in ids8]))

    GATE1["page8"] = dict(reproduced=bool(worst < 1e-9), max_abs_drift=worst,
                          config_id=PAIRED_CONFIG, verifier=VERIFIER)
    print(f"\nmax |recomputed - shipped|: {worst:.2e}   "
          f"-> {'REPRODUCED' if GATE1['page8']['reproduced'] else 'DRIFTED'}")
else:
    GATE1["page8"] = dict(reproduced=False, skipped=True)
    print("page 8 traces or scores not on disk; gate 1c cannot run")

## 7. The family chart: accuracy against measured inference FLOPs

Every point above, on one pair of axes and one FLOPs rule. The chart the page's section 1
currently *authors*, here measured — the evidence for the spec's B1+B2.

In [ ]:
FAMILY_CHART = None

if all(GATE1.get(k, {}).get("reproduced") for k in ("page6", "page7", "page8")):
    top_budget = max(P6_POINTS[BIG])
    greedy_pts = []
    for mid in FAMILY:
        pt = P6_POINTS[mid][top_budget]
        greedy_pts.append(dict(
            model_id=mid, label=mid.split("/")[-1].replace("-Instruct", ""),
            params=PARAMS[mid], accuracy=pt["accuracy"], mean_tokens=pt["mean_tokens"],
            flops=FLOPS(PARAMS[mid], pt["mean_tokens"]),
            source=f"page 6, greedy, cap {top_budget}"))
    target = next(g for g in greedy_pts if g["model_id"] == BIG)

    vote_pts = [dict(n=pt["n"], accuracy=pt["accuracy"], coverage=pt["coverage"],
                     mean_tokens=pt["mean_tokens"],
                     flops=FLOPS(PARAMS[PRIMARY], pt["mean_tokens"]),
                     source=f"page 7, T={T_VOTE} pile")
                for pt in P7_VOTE_CURVE]

    # The judge is the same lineage's 1.5B body; its pass over each candidate is charged
    # at the same 2*P*tokens rule. The question prefix is omitted on both sides.
    judge_params = PARAMS["Qwen/Qwen2.5-1.5B-Instruct"]
    gen_flops = FLOPS(PARAMS[PRIMARY], P8_RUNG["mean_tokens"])
    judge_flops = FLOPS(judge_params, P8_RUNG["mean_tokens"])
    best_of = dict(**P8_RUNG, gen_flops=gen_flops, judge_flops=judge_flops,
                   flops=gen_flops + judge_flops, judge_params=judge_params,
                   source="page 8, its own pool")

    FAMILY_CHART = dict(band="math500_l45", n_items=HARD_N, target=target,
                        greedy=greedy_pts, vote_curve=vote_pts, best_of=best_of,
                        flops_rule="2 * params * tokens_generated, measured tokens; "
                                   "prompt prefix omitted on every point")

    print(f"{'point':<34}{'accuracy':>9}{'FLOPs':>12}{'x target':>10}")
    for g in greedy_pts:
        tag = "   <- the target" if g is target else ""
        print(f"{g['label'] + ', greedy':<34}{g['accuracy']:>9.0%}"
              f"{g['flops']:>12.2e}{g['flops'] / target['flops']:>10.2f}{tag}")
    for v in vote_pts:
        print(f"{f'3B, vote of {v['n']} (T={T_VOTE})':<34}{v['accuracy']:>9.0%}"
              f"{v['flops']:>12.2e}{v['flops'] / target['flops']:>10.2f}")
    print(f"{'3B, best-of-16 (' + best_of['selector'].split('(')[1].rstrip(')') + ')':<34}"
          f"{best_of['accuracy']:>9.0%}{best_of['flops']:>12.2e}"
          f"{best_of['flops'] / target['flops']:>10.2f}   incl. the judge's pass")
    print(f"{'3B, coverage@16 (no realizable rule)':<34}"
          f"{vote_pts[-1]['coverage']:>9.0%}{'-':>12}{'-':>10}")
else:
    print("gate 1 incomplete; the family chart is not derived from unverified inputs")

In [ ]:
import matplotlib.pyplot as plt

if FAMILY_CHART:
    F = FAMILY_CHART
    tgt = F["target"]
    fig, ax = plt.subplots(figsize=(8.8, 5.0))

    ax.axhline(tgt["accuracy"] * 100, color="#8a8578", lw=1.2, ls=(0, (4, 3)))
    ax.annotate("the target: 7B, one greedy answer",
                (F["vote_curve"][0]["flops"], tgt["accuracy"] * 100),
                textcoords="offset points", xytext=(0, 5), fontsize=8, color="#8a8578")

    hues = {"Qwen2.5-1.5B": "#4a7a5c", "Qwen2.5-3B": "#3b4a6b", "Qwen2.5-7B": "#1a1a1a"}
    for g in F["greedy"]:
        ax.scatter([g["flops"]], [g["accuracy"] * 100], s=70, zorder=5,
                   color=hues[g["label"]], edgecolor="white", linewidth=1.2)
        ax.annotate(f"{g['label']}\ngreedy", (g["flops"], g["accuracy"] * 100),
                    textcoords="offset points", xytext=(8, -4), fontsize=8,
                    color=hues[g["label"]])

    xs = [v["flops"] for v in F["vote_curve"]]
    ax.plot(xs, [v["accuracy"] * 100 for v in F["vote_curve"]], marker="o", ms=4.5,
            lw=2.2, color="#3b4a6b", label="3B, majority vote of N (page 7)")
    ax.plot(xs, [v["coverage"] * 100 for v in F["vote_curve"]], ls="--", lw=1.4,
            color="#a2704a", label="3B, a right answer is in the pile (no rule reaches it)")
    for v in (F["vote_curve"][0], F["vote_curve"][-1]):
        ax.annotate(f"N={v['n']}", (v["flops"], v["accuracy"] * 100),
                    textcoords="offset points", xytext=(0, -14), fontsize=8,
                    ha="center", color="#3b4a6b")

    b = F["best_of"]
    ax.scatter([b["flops"]], [b["accuracy"] * 100], s=90, marker="D", zorder=5,
               color="#b23", edgecolor="white", linewidth=1.2,
               label=f"3B, best-of-{b['n']} under the trained verifier (page 8), "
                     f"judge's pass included")

    ax.set_xscale("log")
    ax.set_xlabel(f"inference FLOPs per problem ({F['flops_rule']})")
    ax.set_ylabel("share of problems solved (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.set_title(f"MATH-500 L4/5, n={F['n_items']}: what the same FLOPs buy, measured "
                 f"(pages 6-8's runs)", fontsize=9, loc="left", color="#555")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_1_family.png", dpi=160)
    plt.show()

    print(textwrap.dedent(f"""
    Read left to right: the 7B's one greedy answer ({tgt['accuracy']:.0%}) sits at
    {tgt['flops']:.1e} FLOPs. Everything the 3B can throw at the problem -- {POOL_N} chains
    and a vote ({F['vote_curve'][-1]['accuracy']:.0%}), or {POOL_N} chains and a trained
    verifier ({b['accuracy']:.0%}) -- lands at or below that line, at
    {F['vote_curve'][-1]['flops'] / tgt['flops']:.1f}-{b['flops'] / tgt['flops']:.1f}x its
    bill. The dashed coverage line is the part the page must not soften: the right answer
    was in the pile on {F['vote_curve'][-1]['coverage']:.0%} of problems. On this band the
    small model does not lack answers; every realizable selector fails to cash them."""))

## 8. Gate 1, the verdict

Go/no-go for everything below. A drift here is not a page-11 problem — it means two pages of
the deck disagree about the same run, which is worth more than this notebook.

In [ ]:
GATE1["all"] = all(GATE1.get(k, {}).get("reproduced") for k in ("page6", "page7", "page8"))
for k in ("page6", "page7", "page8"):
    g = GATE1.get(k, {})
    print(f"  {k:<8} {'REPRODUCED' if g.get('reproduced') else 'FAILED':<12}"
          + (f"max drift {g['max_abs_drift']:.2e}" if "max_abs_drift" in g else "(skipped)"))
print(f"\ngate 1: {'PASS -- the derivation below stands on verified inputs'
                   if GATE1['all'] else 'FAIL -- stop here; fix the drift first'}")
assert GATE1["all"] or RUN_MODE == "REPLAY", (
    "gate 1 failed; QUICK and FULL may not spend compute on unverified inputs")

## 9. Generation, page 7's machinery

Used by gate 2 and the arms. A chain that hits the cap is force-committed exactly as pages
6–8 did it, and marked `forced`, so a broken step is always attributable — to us or to the
model.

In [ ]:
FORCE_STEM = "\n\nFinal answer: \\boxed{"

model = tokenizer = device = None
EOS_IDS = set()
DTYPE = None


def load_generator(model_id, system=None):
    """One checkpoint at a time; the arms sweep sequentially on one card."""
    global model, tokenizer, device, EOS_IDS, DTYPE
    if model is not None:
        del model
        harness.free_memory()
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    # The 7B in bf16 (~15 GB) OOMs a 16 GB card; page 6 loaded the 7B target in
    # 8-bit (load_in_8bit=(quant=="8bit")), so match it for instrument continuity.
    _eight = device.type == "cuda" and "7B" in model_id
    loaded = harness.load_model(model_id, device=device, dtype=DTYPE, load_in_8bit=_eight)
    model, tokenizer = loaded["model"], loaded["tokenizer"]
    EOS_IDS = set(harness.end_of_turn_ids(tokenizer))
    harness.selftest(tokenizer, verbose=False)
    return dict(id=model_id, params=loaded["params"], dtype=str(loaded["dtype"]),
                system_prompt=system)


def chat_for(question, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) \
        + [{"role": "user", "content": render_prompt(question)}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def cut_at_boxed_close(pieces):
    """Page 7's: walk the forced continuation until our opened brace closes."""
    depth, kept = 1, []
    for p in pieces:
        out = ""
        for ch in p:
            out += ch
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    kept.append(out)
                    return kept, True
        kept.append(out)
    return kept, False


def group_seed(problem_id, cid, group_index):
    return int(harness.sha256_of([str(SEED), problem_id, cid, str(group_index)])[:15], 16)


@torch.inference_mode()
def sample_group(question, cap, temperature, k, seed, system=None,
                 allowance=ANSWER_ALLOWANCE):
    """k chains of one problem in one generate call. Page 7's, plus the system slot the
    Math model's card asks for."""
    if temperature == 0.0 and k > 1:
        out = []
        for _ in range(k):
            out.extend(sample_group(question, cap, temperature, 1, seed, system, allowance))
        return out

    chat = chat_for(question, system)
    prompt_ids = tokenizer(chat, add_special_tokens=False).input_ids
    inp = torch.tensor([prompt_ids], device=model.device)
    sampling = temperature > 0
    if sampling:
        torch.manual_seed(seed)
    out = model.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=cap,
        do_sample=sampling,
        temperature=temperature if sampling else None,
        top_p=TOP_P if sampling else None, top_k=None,
        num_return_sequences=k,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id)

    chains = []
    for row in out:
        gen = row[inp.shape[1]:].tolist()
        stop = next((i for i, t in enumerate(gen) if t in EOS_IDS), None)
        hit_cap = stop is None
        gen = gen[:stop] if stop is not None else gen[:cap]

        own_text = tokenizer.decode(gen, skip_special_tokens=True)
        own_pieces = harness.incremental_token_strings(gen, tokenizer)
        assert "".join(own_pieces) == own_text

        if not hit_cap:
            chains.append(dict(trace=own_text, tokens=own_pieces, own_tokens=len(gen),
                               answer_tokens=0, tokens_generated=len(gen),
                               truncated=False, forced=False, forced_suffix="",
                               answer_closed=None, cut_index=None, answer_start=None))
            continue

        suffix_ids = tokenizer(FORCE_STEM, add_special_tokens=False).input_ids
        suffix_pieces = harness.incremental_token_strings(suffix_ids, tokenizer)
        assert "".join(suffix_pieces) == FORCE_STEM
        forced_inp = torch.tensor([prompt_ids + gen + suffix_ids], device=model.device)
        ans = model.generate(input_ids=forced_inp,
                             attention_mask=torch.ones_like(forced_inp),
                             max_new_tokens=allowance, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tokenizer.pad_token_id
                             if tokenizer.pad_token_id is not None
                             else tokenizer.eos_token_id)
        ans_ids = ans[0][forced_inp.shape[1]:].tolist()
        while ans_ids and ans_ids[-1] in EOS_IDS:
            ans_ids.pop()
        kept, closed = cut_at_boxed_close(harness.incremental_token_strings(ans_ids,
                                                                            tokenizer))
        tokens = own_pieces + suffix_pieces + kept
        chains.append(dict(trace="".join(tokens), tokens=tokens, own_tokens=len(gen),
                           answer_tokens=len(kept),
                           tokens_generated=len(gen) + len(kept),
                           truncated=True, forced=True, forced_suffix=FORCE_STEM,
                           answer_closed=closed, cut_index=len(own_pieces),
                           answer_start=len(own_pieces) + len(suffix_pieces)))
    return chains


def config_id(temperature, cap):
    return f"t{temperature:.2f}-c{cap}"


print("generation machinery ready (loads nothing until a gate or an arm asks)")

## 10. Gate 2 — the specialist smoke test (`QUICK` and up)

Pre-registered: 20 hard-band items, greedy. Two numbers come out — accuracy against the
floor, and mean chain length, which is what turns the 4.9× parameter ratio into a *measured*
FLOPs ratio or refuses to. Below the floor, arm B does not run and the spec's branch 5
applies.

In [ ]:
GATE2 = None

if RUN_MODE in ("QUICK", "FULL"):
    meta_math = load_generator(MATH_MODEL, system=MATH_SYSTEM)
    print(f"loaded {MATH_MODEL} ({meta_math.get('params', 0):,} params)\n")

    smoke = []
    for row in harness.progress(items_hard[:GATE2_ITEMS], desc="gate 2", unit="item"):
        ch = sample_group(row["problem"], CHAIN_CAP, 0.0, 1,
                          group_seed(row["unique_id"], config_id(0.0, CHAIN_CAP), 0),
                          system=MATH_SYSTEM)[0]
        cand = extract_final_candidate(ch["trace"])
        ch.update(problem_id=row["unique_id"], level=int(row["level"]),
                  ground_truth=row["answer"], extracted_answer=cand,
                  correct=bool(cand is not None and grade_answer(cand, row["answer"])))
        smoke.append(ch)

    acc = mean([c["correct"] for c in smoke])
    tok = mean([c["tokens_generated"] for c in smoke])
    p7b = P6_POINTS[BIG][max(P6_POINTS[BIG])] if 6 in SRC else None
    GATE2 = dict(n=len(smoke), accuracy=acc, mean_tokens=tok,
                 passed=bool(acc >= GATE2_MIN), floor=GATE2_MIN,
                 params=meta_math.get("params"))
    print(f"\n{MATH_MODEL.split('/')[-1]}, greedy, {len(smoke)} hard-band items")
    print(f"  accuracy      {acc:.0%}   (pre-registered floor {GATE2_MIN:.0%})")
    print(f"  mean tokens   {tok:,.0f}"
          + (f"   (the 7B's greedy chains: {p7b['mean_tokens']:,.0f})" if p7b else ""))
    if GATE2["params"] and p7b:
        f_ratio = FLOPS(PARAMS[BIG], p7b["mean_tokens"]) / FLOPS(GATE2["params"], tok)
        print(f"  measured FLOPs ratio, 7B greedy / Math greedy: {f_ratio:.1f}x")
    print(f"\ngate 2: {'PASS -- arm B runs' if GATE2['passed'] else
                       'FAIL -- arm B does not run; the spec' + chr(39) + 's branch 5 applies'}")
else:
    print("REPLAY: gate 2 needs a model and does not run here")

## 11. Arms A and B (`FULL`)

Arm A: the generalist family on the easy band. Arm B: the specialist on both bands. One
trace log, pages 5–8's schema, resumable; the plan below prices the run from measured spends
before anything is generated.

In [ ]:
# The work list: (model_id, system, band_name, items, temperature, k)
ARM_PLAN = [
    ("Qwen/Qwen2.5-1.5B-Instruct", None, "easy", "items_easy", 0.0, 1),
    ("Qwen/Qwen2.5-1.5B-Instruct", None, "easy", "items_easy", T_VOTE, POOL_N),
    ("Qwen/Qwen2.5-3B-Instruct",   None, "easy", "items_easy", 0.0, 1),
    ("Qwen/Qwen2.5-3B-Instruct",   None, "easy", "items_easy", T_VOTE, POOL_N),
    ("Qwen/Qwen2.5-7B-Instruct",   None, "easy", "items_easy", 0.0, 1),
    (MATH_MODEL, MATH_SYSTEM, "easy", "items_easy", 0.0, 1),
    (MATH_MODEL, MATH_SYSTEM, "easy", "items_easy", T_VOTE, POOL_N),
    (MATH_MODEL, MATH_SYSTEM, "hard", "items_hard", 0.0, 1),
    (MATH_MODEL, MATH_SYSTEM, "hard", "items_hard", T_VOTE, POOL_N),
]

_bands = {"items_easy": items_easy, "items_hard": items_hard}
n_chains = sum(len(_bands[b]) * k for _, _, _, b, _, k in ARM_PLAN)
print(f"the plan: {len(ARM_PLAN)} sweeps, {n_chains:,} chains at cap {CHAIN_CAP}")
for mid, _, band, b, t, k in ARM_PLAN:
    print(f"  {mid.split('/')[-1]:<28} {band:<5} T={t:<4} k={k:<3} "
          f"({len(_bands[b]) * k:,} chains)")

if RUN_MODE == "FULL":
    assert GATE2 and GATE2["passed"] or not any(m == MATH_MODEL for m, *_ in ARM_PLAN), \
        "gate 2 failed; strike the Math sweeps from ARM_PLAN to run arm A alone"
    RUN_ID = (f"p11-{RUN_MODE.lower()}-e{EASY_HASH[:8]}-h{HARD_HASH[:8]}"
              f"-cap{CHAIN_CAP}-pool{POOL_N}-p{TOP_P}-{str(DTYPE).rsplit('.', 1)[-1]}"
              f"-{device.type}")
    STORE = harness.TraceStore(TRACES_PATH, RUN_ID,
                               ("model_id", "config_id", "problem_id", "sample_index"))
    STORE.report()

    current = MATH_MODEL if model is not None else None
    for mid, system, band, bname, t, k in ARM_PLAN:
        if mid != current:
            meta_m = load_generator(mid, system=system)
            current = mid
        cid = config_id(t, CHAIN_CAP)
        rows = _bands[bname]
        bar = harness.progress(rows, desc=f"{mid.split('/')[-1]} {band} {cid}", unit="item")
        for row in bar:
            pid = row["unique_id"]
            got = [STORE.get(mid, cid, pid, s) for s in range(k)]
            step = 1 if t == 0.0 else GROUP
            for start in range(0, k, step):
                size = min(step, k - start)
                if all(got[s] is not None for s in range(start, start + size)):
                    continue
                g_index = start // step
                seed = group_seed(pid, cid, g_index)
                made = sample_group(row["problem"], CHAIN_CAP, t, size, seed, system)
                for j, g in enumerate(made):
                    s = start + j
                    cand = extract_final_candidate(g["trace"])
                    got[s] = STORE.append({
                        "model_id": mid, "config_id": cid, "problem_id": pid,
                        "sample_index": s, "group_index": g_index, "batch_seed": seed,
                        "batch_size": size, "band": band,
                        "level": int(row["level"]), "subject": row.get("subject"),
                        "question": row["problem"], "ground_truth": row["answer"],
                        "temperature": t, "top_p": TOP_P if t > 0 else None,
                        "chain_budget": CHAIN_CAP, "system_prompt": system,
                        **{kk: g[kk] for kk in
                           ("trace", "tokens", "own_tokens", "answer_tokens",
                            "tokens_generated", "truncated", "forced", "forced_suffix",
                            "answer_closed", "cut_index", "answer_start")},
                        "extracted_answer": cand,
                        "vote_key": vote_key(g["trace"]),
                        "correct": bool(cand is not None
                                        and grade_answer(cand, row["answer"]))})
    print("\narms complete")
else:
    print(f"\n{RUN_MODE}: nothing generated; flip RUN_MODE to FULL for the arms")

## 12. What the arms say (when they exist)

The same estimator as arm 0, on the new traces: per band, every model's greedy point and
vote curve against the 7B's greedy answer on that band — and the branch, read off the
pre-registered table in the spec rather than argued after the fact.

In [ ]:
ARMS = None
BRANCH = None

if TRACES_PATH.exists():
    new_raw = read_traces(TRACES_PATH)
    S11 = {}
    for r in new_raw:
        S11.setdefault((r["model_id"], r["band"], r["config_id"]),
                       {}).setdefault(r["problem_id"], []).append(r)
    for k in S11:
        for pid in S11[k]:
            S11[k][pid].sort(key=lambda r: r["sample_index"])

    def band_points(band_name, band_items):
        pts = []
        for (mid, band, cid), by_pid in sorted(S11.items(), key=lambda kv: str(kv[0])):
            if band != band_name:
                continue
            t = float(cid.split("-")[0][1:])
            params = PARAMS.get(mid) or (GATE2 or {}).get("params")
            if t == 0.0:
                rs = [by_pid[p["unique_id"]][0] for p in band_items
                      if p["unique_id"] in by_pid]
                acc = mean([r["correct"] for r in rs])
                tok = mean([r["tokens_generated"] for r in rs])
                pts.append(dict(model_id=mid, kind="greedy", n=1, accuracy=acc,
                                mean_tokens=tok, n_items=len(rs), params=params,
                                flops=FLOPS(params, tok) if params else None))
            else:
                for n in N_AXIS:
                    cp = curve_point(by_pid, band_items, n)
                    if cp:
                        pts.append(dict(model_id=mid, kind="vote", temperature=t,
                                        params=params, **cp,
                                        flops=FLOPS(params, cp["mean_tokens"])
                                        if params else None))
        return pts

    ARMS = {"easy": band_points("easy", items_easy),
            "hard": band_points("hard", items_hard)}

    for band_name, band_items in (("easy", items_easy), ("hard", items_hard)):
        pts = ARMS[band_name]
        if not pts:
            continue
        tgt7 = next((p for p in pts if p["model_id"] == BIG and p["kind"] == "greedy"),
                    None)
        print(f"\n{band_name} band, n={len(band_items)}"
              + (f"   target: 7B greedy {tgt7['accuracy']:.0%}" if tgt7 else ""))
        for p in pts:
            name = p["model_id"].split("/")[-1].replace("-Instruct", "")
            what = "greedy" if p["kind"] == "greedy" else f"vote@{p['n']}"
            x = f"{p['flops']:.2e}" if p.get("flops") else "-"
            print(f"  {name:<22}{what:<10}{p['accuracy']:>7.0%}{x:>12}")

    # ---- the branch, per the spec's pre-registered table -------------------------
    if RUN_MODE == "FULL" and ARMS["easy"] and ARMS["hard"]:
        tgt_easy = next(p for p in ARMS["easy"]
                        if p["model_id"] == BIG and p["kind"] == "greedy")
        crossing_easy = any(
            p["kind"] == "vote" and p["model_id"] != BIG and p.get("flops")
            and p["accuracy"] >= tgt_easy["accuracy"]
            and p["flops"] < FLOPS(PARAMS[BIG], tgt_easy["mean_tokens"])
            for p in ARMS["easy"])
        tgt_hard = FAMILY_CHART["target"] if FAMILY_CHART else None
        math_hard = [p for p in ARMS["hard"] if p["model_id"] == MATH_MODEL]
        math_greedy = next((p for p in math_hard if p["kind"] == "greedy"), None)
        beats = ties = None
        if math_greedy and tgt_hard:
            beats = math_greedy["accuracy"] > tgt_hard["accuracy"]
            ties = abs(math_greedy["accuracy"] - tgt_hard["accuracy"]) < 0.03
        BRANCH = dict(crossing_on_easy_band=crossing_easy,
                      specialist_beats_target=beats, specialist_ties_target=ties,
                      branch=(5 if beats is False and not ties else
                              (2 if crossing_easy else 1) if beats else
                              (4 if crossing_easy else 3)))
        print(f"\npre-registered branch: {BRANCH['branch']}   "
              f"(crossing on easy: {crossing_easy}, specialist beats target: {beats})")
        print("the page treatment for this branch is spec section 6, verbatim")
else:
    print("no page-11 traces yet; the arms have not run")

## 13. Artifacts

In [ ]:
manifest = {
    "run_id": (f"p11-arm0-h{HARD_HASH[:8]}" if RUN_MODE == "REPLAY"
               else RUN_ID if RUN_MODE == "FULL" else f"p11-quick-h{HARD_HASH[:8]}"),
    "notebook": "page_11_training_vs_scale.ipynb",
    "notebook_sha256": harness.file_sha256("page_11_training_vs_scale.ipynb"),
    "harness": harness.fingerprint(),
    "page": 11, "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "spec": "docs/page_11_refactor_spec.md",
    "flops_rule": "2 * params * tokens_generated, measured tokens; prompt prefix omitted "
                  "on every point; a best-of point adds its judge's pass at the same rule",
    "sources": {str(n): dict(run_id=(SRC[n]["manifest"].get("run_id")
                                     or SRC[n]["manifest"].get("score_id")),
                             traces_sha256=harness.file_sha256(UP[n] / "traces.jsonl"))
                for n in SRC},
    "items": {"hard": dict(levels=list(HARD_LEVELS), n=HARD_N, sha256=HARD_HASH,
                           same_as_pages_6_7=bool(p7_manifest_path.exists())),
              "easy": dict(levels=list(EASY_LEVELS), n=EASY_N, sha256=EASY_HASH),
              "seed": SEED},
    "specialist": {"model": MATH_MODEL, "system_prompt": MATH_SYSTEM,
                   "provenance": "Qwen2.5-Math report: continued pretraining on the Qwen "
                                 "Math Corpus v2 (>1T tokens), SFT on rejection-sampled "
                                 "rationales incl. the MATH training split, GRPO with a "
                                 "rule-based correctness verifier in the reward. The "
                                 "attribution is the bundle; no ablation separates it."},
    "decoding": {"top_p": TOP_P, "pool_n": POOL_N, "vote_temperature": T_VOTE,
                 "chain_cap": CHAIN_CAP, "group_size": GROUP,
                 "seed_rule": "sha256(SEED, problem_id, config_id, group_index)",
                 "answer_allowance": ANSWER_ALLOWANCE},
    "estimator": {"n_axis": N_AXIS, "n_resamples": N_RESAMPLES,
                  "rule": "page 7's: mean over random N-subsets, page 7's seeded draws"},
    "gates": {"gate1": GATE1,
              "gate2": GATE2 or {"floor": GATE2_MIN, "n": GATE2_ITEMS, "ran": False}},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

derived = {
    "manifest": manifest,
    # Read before any number on this page.
    "claims": {
        # Arm 0's whole standing: the shipped numbers of pages 6-8, re-derived here,
        # to the last bit. False means NOTHING on this page may be quoted.
        "gate1_reproduced": GATE1["all"],
        # No vote or verifier configuration of the 3B reaches the 7B's greedy answer
        # for fewer FLOPs on the hard band (spec B1, hard half).
        "no_crossing_on_hard_band": bool(FAMILY_CHART and not any(
            v["accuracy"] >= FAMILY_CHART["target"]["accuracy"]
            and v["flops"] < FAMILY_CHART["target"]["flops"]
            for v in FAMILY_CHART["vote_curve"])),
        # The trained verifier's rung lands on the target, at many times its bill.
        "selection_at_best_ties_target": bool(FAMILY_CHART and abs(
            FAMILY_CHART["best_of"]["accuracy"]
            - FAMILY_CHART["target"]["accuracy"]) < 1e-9),
        # Coverage clears the target: the failure is selection, not absence (spec B2).
        "coverage_clears_target": bool(FAMILY_CHART and
                                       FAMILY_CHART["vote_curve"][-1]["coverage"]
                                       > FAMILY_CHART["target"]["accuracy"]),
        "gate2_ran": bool(GATE2),
        "gate2_passed": GATE2["passed"] if GATE2 else None,
        "arms_run": bool(ARMS and RUN_MODE == "FULL"),
    },
    "family": FAMILY_CHART,
    "arms": ARMS,
    "branch": BRANCH,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False),
                        encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False),
                         encoding="utf-8")
print(f"derived.json    {harness.kb(DERIVED_PATH):>9,.1f} KB   <- bundled with the page")
print(f"manifest.json   {harness.kb(MANIFEST_PATH):>9,.1f} KB")
if TRACES_PATH.exists():
    print(f"traces.jsonl    {harness.kb(TRACES_PATH):>9,.1f} KB   <- download tier")
print()
for k, v in derived["claims"].items():
    print(f"  {k:<34} {v}")

## Notes for re-derivation

- **Arm 0 generates nothing.** Its every number is a re-aggregation of pages 6–8's
  published `traces.jsonl` (and page 8's `scores.jsonl`), and gate 1 asserts the
  re-aggregation equals what those pages shipped. To re-derive from scratch: unpack the
  three published zips so their `out/` directories sit beside this notebook, and run
  `REPLAY`.
- **The FLOPs axis is measured tokens times parameters, twice.** No point uses a cap or a
  parameter *ratio* as a stand-in; the caption states the omission of the prompt prefix,
  which is shared by every point and does not reorder any of them.
- **The branch is read, not argued.** Section 12 maps the arms' results onto the spec's
  pre-registered table (section 6) and names the branch number; the page treatment for
  every branch was written before the first new chain was generated.